# Konu 8 — Koddan görsel üretimi

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 7'de `ciz("...")` yazdın, `ciktilar/001.jpg` çıktı. Arada ne oldu? Bugün o arayı **kendimiz yazıyoruz**:
isteği göndermek, gelen görseli dosyaya yazmak, dosyaya düzenli bir ad vermek, hataları yakalamak.

Bugün birkaç hücre **bilerek** hata verir: kırmızı yazı orada bir deneyin sonucudur; oku ve devam et.

Isınma anahtarsız. Adım 1'den sonrası `anahtar.txt` ister (Konu 2'deki dosya). Defterin tamamı 5 görsel üretir.

## Isınma — Görsel de bir dosya, ama metin değil

Konu 2'de cevapları `"w"` ile bir metin dosyasına yazmıştık. Bu klasörde `veri/fincan.jpg` adında bir görsel var.

**Tahmin et, sohbete yaz:** görseli, Konu 2'de metin dosyasını açtığımız gibi açıp okursak Python ne yapar?
Anlamsız harfler mi gösterir, hata mı verir, başka bir şey mi?

In [ ]:
with open("veri/fincan.jpg", encoding="utf-8") as dosya:
    icerik = dosya.read()

`UnicodeDecodeError`: Python dosyadaki sayıları harflere çevirmeye çalıştı ve çeviremedi. Çünkü bir görsel harflerden değil,
**baytlardan** oluşur: 0 ile 255 arasında sayılar (Konu 3'teki RGB değerleriyle aynı aralık; ama JPEG'de bunlar renkler değil,
sıkıştırılmış veri). Bu tür dosyaya **ikili dosya** (binary) denir.

İkili dosyayı açmanın yolu `"rb"`: **r**ead (oku) + **b**inary (ikili). `encoding` yazılmaz; harfe çevirme yok.

In [ ]:
with open("veri/fincan.jpg", "rb") as dosya:
    baytlar = dosya.read()

print(len(baytlar))
print(baytlar[:12])

Görsel 9579 bayt. Başındaki `b'...'` "bu bir bayt dizisi" demek; `\xff` gibi yazılanlar harfe karşılık gelmeyen baytlar.
İlk iki bayt (`\xff\xd8`) her JPEG'de aynıdır; `JFIF` yazısı da çoğu JPEG'de bulunur ve "ben bir JPEG'im" der.

## Isınma 2 — Görseli metne çevirmek: base64

Konu 2'deki gibi, API'ler cevabı JSON içinde metin olarak gönderir. Peki görsel nasıl metin olur?
**base64** baytları harf ve rakamlara çevirir (`b64encode`); öbür uçta geri çevrilir (`b64decode`).

In [ ]:
import base64

metin = base64.b64encode(baytlar)
print(len(metin))
print(metin[:40])

9579 bayt 12772 karaktere dönüştü: üçte bir kadar uzadı. Artık yalnızca harf, rakam, `+` ve `/` var; her JPEG'in base64 hâli `/9j/` ile başlar.
Başta hâlâ `b'...'` görünüyor: `b64encode` sonucu bayt olarak verir, ama içinde yalnızca harf ve rakam var.
base64 şifreleme ya da sıkıştırma değildir; dosyayı büyütür, herkes geri çevirebilir.

Geri çevirelim ve asılla karşılaştıralım:

In [ ]:
geri = base64.b64decode(metin)
print(geri == baytlar)

`True`: tek bayt kaybolmadı. Bugün modelden gelen görsel tam olarak böyle gelecek: base64 metni olarak.

## Adım 1 — Anahtar, adres, başlık

Konu 2'nin Adım 1–2'sinin aynısı; yalnızca model değişti: Konu 7'deki `flux` (FLUX.1 schnell).
`base64`'ü burada bir daha alıyoruz ki deftere buradan başlayan da kullanabilsin.

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print("Anahtar okundu")

In [ ]:
import base64
import requests

MODEL = "@cf/black-forest-labs/flux-1-schnell"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}

## Adım 2 — İsteği gönder

Gövdede iki alan var: `prompt` (yönerge) ve `seed` (tohum). Konu 7'deki `ciz(yonerge, tohum=42)` tam olarak bunu gönderiyordu.

In [ ]:
govde = {"prompt": "a ceramic coffee cup on a wooden desk, watercolor illustration", "seed": 42}
cevap = requests.post(adres, headers=basliklar, json=govde)
print(cevap.status_code)

`200` gördüysen istek başarılı. Görsel birkaç saniyede geldi; şimdi nerede olduğunu bulalım.

## Adım 3 — Yanıtın içi

Konu 2'deki gibi önce `result` kutusunu, sonra içindeki alanı açıyoruz. Metin modelinde alanın adı `response` idi;
görsel modelinde `image`. Slaytta API'lerin görseli JSON içinde nasıl gönderdiğini konuştuk. Bakalım `image` alanında ne var:

In [ ]:
yanit = cevap.json()
sonuc = yanit["result"]
metin = sonuc["image"]

print(len(metin))
print(metin[:40])

Çok uzun bir harf dizisi ve `/9j/` ile başlıyor: görsel, base64 metni olarak geldi. (Bu kez başta `b'` yok: JSON'dan gelen her şey düz metin.)

## Adım 4 — Metinden görsele, görselden dosyaya

`b64decode` metni baytlara çevirir:

In [ ]:
resim = base64.b64decode(metin)
print(len(resim))

Baytları dosyaya yazacağız. **Tahmin et:** Konu 2'deki gibi `"w"` ile yazarsak ne olur?

In [ ]:
with open("deneme.jpg", "w") as dosya:
    dosya.write(resim)

`TypeError: write() argument must be str, not bytes`: `"w"` metin bekler, biz bayt verdik.
(Dosya açıldığı için klasörde boş bir `deneme.jpg` oluştu; silebilirsin.)

Baytları yazmanın yolu `"wb"`: **w**rite (yaz) + **b**inary (ikili).

In [ ]:
with open("ilk.jpg", "wb") as dosya:
    dosya.write(resim)

Klasörde `ilk.jpg` oluştu; VS Code'da tıklayıp açabilirsin. Defterde göstermek için Konu 6'daki `imshow`'u kullanıyoruz;
`plt.imread` bir görsel dosyasını okuyup sayı tablosuna çevirir.

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(plt.imread("ilk.jpg"))
plt.axis("off")
plt.show()

Konu 7'de `ciz` yazınca arka planda olan buydu: metin → bayt → dosya.

## Adım 5 — Fonksiyona koy: `ciz`

Adım 2–4'ü bir fonksiyona topluyoruz (Konu 2'deki `modele_sor` gibi). Bir fark var: durum koduna bakıyoruz.

- İstek başarısızsa fonksiyon bir mesaj yazar ve `None` ("hiçbir şey") döndürür.
- `return` fonksiyonu **orada bitirir**: alttaki `cevap.json()` satırları hiç çalışmaz. Böylece program, içinde görsel
  olmayan bir yanıtı açmaya çalışıp çökmez.
- Fonksiyon görselin **baytlarını** geri verir; dosyaya yazmak başka bir fonksiyonun işi olacak.
- Konu 7'deki `ciz` kaydedip gösteriyordu ve tohum zorunlu değildi; bizimki şimdilik yalnızca baytları veriyor,
  kaydetmeyi Adım 7'de ayrı yazacağız.

In [ ]:
def ciz(yonerge, tohum):
    govde = {"prompt": yonerge, "seed": tohum}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    if cevap.status_code != 200:
        print("İstek başarısız, durum kodu:", cevap.status_code)
        return None
    yanit = cevap.json()
    sonuc = yanit["result"]
    metin = sonuc["image"]
    return base64.b64decode(metin)

In [ ]:
resim = ciz("a red bicycle leaning on a yellow wall, flat vector poster", 7)
print(len(resim))

Fonksiyon görselin baytlarını geri verdi; `print` bayt sayısını yazdı. Henüz dosyaya yazmadık.

## Adım 6 — Düzenli klasör, düzenli ad

"gorsel.jpg", "gorsel2.jpg", "son_son.jpg"... Bir hafta sonra hangisi neydi, bilemezsin. Dosya adı kendini anlatmalı:
**ne zaman**, **hangi yönergeyle**, **hangi tohumla**.

`Path` (pathlib) dosya ve klasör yollarıyla çalışır. `mkdir` klasörü oluşturur; `exist_ok=True` "zaten varsa sorun yok" demek.

In [ ]:
from pathlib import Path
from datetime import datetime

klasor = Path("ciktilar")
klasor.mkdir(exist_ok=True)

Adı üç parçadan kuruyoruz. Önce **tarih**: `datetime.now()` şu anki tarih ve saat; `strftime` onu istediğimiz biçimde yazar.
`%Y` yıl, `%m` ay, `%d` gün, `%H` saat, `%M` dakika, `%S` saniye. Dikkat: küçük `m` ay, büyük `M` dakika.
Yıl başta olduğu için dosyalar adlarına göre sıralanınca tarih sırasına da girer.

In [ ]:
simdi = datetime.now()
tarih = simdi.strftime("%Y-%m-%d_%H-%M-%S")
print(tarih)

Sonra **yönergenin başı**: `yonerge[:20]` ilk 20 karakteri alır; `replace(" ", "-")` boşlukları tireye çevirir.
Bizim kod yalnızca boşluğu düzeltiyor; yönergeye `/` koyma (Python onu alt klasör sanar). Windows'ta `\ : ? * " < > |` da dosya adında hata verir.

In [ ]:
yonerge = "a red bicycle leaning on a yellow wall, flat vector poster"
tohum = 7
ozet = yonerge[:20].replace(" ", "-")
ad = f"{tarih}_{ozet}_{tohum}.jpg"
print(ad)

Son olarak klasörle birleştiriyoruz: `Path`'te `/` "bu klasörün içindeki bu dosya" demek.

In [ ]:
yol = klasor / ad
print(yol)

Yazmadan önce bakıyoruz: bu adda bir dosya **zaten var mı**? Varsa üzerine yazmıyoruz; önceki görsel kaybolmasın.

In [ ]:
if yol.exists():
    print("Bu ad alınmış, üzerine yazmıyorum:", yol)
else:
    with open(yol, "wb") as dosya:
        dosya.write(resim)
    print("Kaydedildi:", yol)

Bu hücreyi **bir kez daha** çalıştır. Ne yazıyor? `yol` değişmedi, dosya artık var: üzerine yazılmadı.

## Adım 7 — Hepsi bir arada: yönergeyi kullanıcıdan al

Adım 6'yı da bir fonksiyona koyuyoruz (Adım 6'nın satırlarını kopyalayıp içeri al): `kaydet` baytları, yönergeyi ve tohumu alır;
dosyanın yolunu geri verir. Ad alınmışsa `None` döndürür ve hiçbir şeyi silmez. Adda saniye olduğu için çoğu zaman tetiklenmez;
ama aynı saniyede iki kayıt olursa ya da bir gün adı tarihsiz kurarsan (Konu 7'deki `001.jpg` gibi) bu satır seni korur.

In [ ]:
def kaydet(resim, yonerge, tohum):
    simdi = datetime.now()
    tarih = simdi.strftime("%Y-%m-%d_%H-%M-%S")
    ozet = yonerge[:20].replace(" ", "-")
    yol = klasor / f"{tarih}_{ozet}_{tohum}.jpg"
    if yol.exists():
        print("Bu ad alınmış, üzerine yazmıyorum:", yol)
        return None
    with open(yol, "wb") as dosya:
        dosya.write(resim)
    return yol

`input` GİTA2112'den tanıdık: kullanıcıya bir soru sorar, yazdığını metin olarak verir. VS Code'da ekranın üstünde bir kutu açılır;
yönergeni İngilizce yaz, **Enter**'a bas.

`resim is not None`: "resim boş değilse", yani istek başarılıysa.

In [ ]:
yonerge = input("Yönerge (İngilizce): ")
tohum = 1
resim = ciz(yonerge, tohum)
if resim is not None:
    yol = kaydet(resim, yonerge, tohum)
    print(yol)

`ciktilar/` klasörüne bak: dosya adı görselin ne zaman, hangi yönergeyle ve hangi tohumla üretildiğini söylüyor.
Konu 7'de elle tuttuğumuz üretim günlüğünün bir kısmı artık dosya adında.

## Adım 8 — Hata yakalama: cevap hiç gelmezse

İki tür başarısızlık var:

- **Sunucu cevap verdi ama "olmaz" dedi** (401 anahtar yanlış, 429 çok hızlı istek): `ciz` bunu durum koduyla yakalıyor.
- **Cevap hiç gelmedi** (internet yok, adres yanlış): `requests.post` bir hata fırlatır ve program **durur**.

İkincisini görmek için isteği var olmayan bir adrese gönderiyoruz (`.invalid` diye bir alan adı yoktur).
Asıl `adres`'e dokunmuyoruz; bozuk adres ayrı bir değişkende. Uzun bir kırmızı yazı çıkacak: **yalnızca en alttaki satırı oku**.

In [ ]:
bozuk_adres = "https://api.cloudflare.invalid/ai/run/flux"
cevap = requests.post(bozuk_adres, headers=basliklar, json=govde)

En alttaki satır `ConnectionError` ile başlıyor: sunucuya ulaşılamadı ve hücre orada durdu. Bir döngünün ortasında olsaydık
bütün iş dururdu.

`try` / `except` ile "dene; şu hata çıkarsa çökme, şunu yap" diyoruz. `except`'ten sonra yazdığımız ad, o en alttaki satırdaki
hatanın adı (`requests` kütüphanesinin hatalarından biri olduğu için başında `requests.exceptions.` var):

In [ ]:
try:
    cevap = requests.post(bozuk_adres, headers=basliklar, json=govde)
except requests.exceptions.ConnectionError:
    print("Sunucuya ulaşılamadı. İnternetini kontrol et, sonra yeniden dene.")

Hata yine oldu ama program durmadı: mesajı yazdı ve devam etti. Gerçek bir araçta `try`'ı `ciz` çağrısının etrafına koyarsın.

## Adım 9 — Var olanı değiştirmek: düzenleme koddan

Bu adımı derste hoca gösterir; sen izle, evde kendin çalıştır.

Konu 7'deki `duzenle` başka bir modelle (FLUX.2 klein) çalışıyordu ve görseli **bir form gibi** gönderiyordu:
yazı alanları (talimat, boyut, tohum) + bir dosya (görsel). Düzenleyici en fazla 512 × 512 piksel kabul ediyor; önce küçültüyoruz.

`PIL` görselleri açıp düzenlemeye yarayan kütüphane (kurulu geliyor). `thumbnail((512, 512))` görseli en-boy oranını koruyarak
512'ye sığdırır; dikkat, sonucu bir değişkene atamıyoruz: görseli **yerinde** değiştirir. Sonra `save` yeni bir dosyaya yazar.

In [ ]:
from PIL import Image

gorsel = Image.open("ilk.jpg")
gorsel.thumbnail((512, 512))
gorsel.save("ilk_kucuk.jpg")

Bu kez `json=` yok ve cevabı ayrı bir değişkende (`duzen_cevap`) tutuyoruz ki Adım 2'nin `cevap`'ıyla karışmasın:

- Yazı alanları `data=` ile gidiyor. Formlar her şeyi metin olarak gönderir; o yüzden sayılar tırnak içinde: `"512"`.
- Dosya `files=` ile gidiyor; `"input_image_0"` modelin ilk girdi görselinin alanı (sayma 0'dan başlıyor).
- Dosya `"rb"` ile açılıyor (Isınma) ve istek `with` bloğunun **içinde** gönderiliyor, çünkü dosya ancak açıkken okunabilir.

In [ ]:
DUZENLEYICI = "@cf/black-forest-labs/flux-2-klein-4b"
duzen_adresi = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{DUZENLEYICI}"
alanlar = {
    "prompt": "make the background a plain sage green color",
    "width": "512",
    "height": "512",
    "seed": "42",
}

with open("ilk_kucuk.jpg", "rb") as dosya:
    dosyalar = {"input_image_0": dosya}
    duzen_cevap = requests.post(duzen_adresi, headers=basliklar, data=alanlar, files=dosyalar)

print(duzen_cevap.status_code)

Üstteki hücre `200` yazmadıysa aşağıdakini çalıştırma: yanıtta görsel yok (Adım 5'teki kural). Durum koduna bak.

`200` ise yanıt yine base64: Adım 3–4'ün aynısı. Bu kez `kaydet` ile düzenli bir ada yazıyoruz.

In [ ]:
yanit = duzen_cevap.json()
sonuc = yanit["result"]
metin = sonuc["image"]
duzenlenmis = base64.b64decode(metin)

yol = kaydet(duzenlenmis, "edit sage green background", 42)
plt.imshow(plt.imread(yol))
plt.axis("off")
plt.show()

Desen hep aynı: **istek → durum kodu → base64 metni → bayt → `"wb"` ile dosya.** Görsel üretimi, düzenleme, ileride video
çoğunlukla bu deseni kullanır.

## Bonus

**1. Tekrar deneme.** Kısa sürede çok istek gönderdiysen (429) yeniden denemek işe yarayabilir. Günlük kota dolduysa
beklemek işe yaramaz; kota ertesi gün yenilenir. `break` döngüyü erkenden bitirir. Bu kod denemeler arasında beklemiyor;
beklemeyi Konu 9'da ekleyeceğiz.

In [ ]:
for deneme in [1, 2, 3]:
    resim = ciz("a paper boat on a puddle, linocut print", 3)
    if resim is not None:
        print("Başarılı, deneme:", deneme)
        break
    print("Olmadı")

if resim is None:
    print("Üç denemede de olmadı; durum koduna bak")

**2. Konu 7'nin aracını oku.** `../07-arayuzden-gorsel/gorsel_araci.py` dosyasını VS Code'da aç. Çoğu parçasını artık tanıyorsun:

| Araçtaki fonksiyon | Bu defterde |
|---|---|
| `anahtari_oku`, `adres_ve_baslik` | Adım 1 |
| `ciz` | Adım 2–5 (kaydetmeyi de kendisi yapıyor; `write_bytes` = `open(..., "wb")` + `write`) |
| `durumu_denetle` | Adım 5'teki durum kodu kontrolü (`None` yerine hata fırlatıyor) |
| `yeni_dosya_adi` | Adım 6 (tarih yerine sıra numarası: `001.jpg`, `002.jpg`...) |
| `resmi_goster` | Adım 4'teki `imshow` |
| `duzenle` | Adım 9 |

Bir istisna: araçtaki `sdxl` modeli görseli base64 metni olarak değil, doğrudan bayt olarak gönderiyor (`cevap.content`).

**3. Konu 9'a hazırlık.** Bu iki fonksiyonla (`ciz`, `kaydet`) 3 stil × 4 renk = 12 görseli nasıl üretirdin?
Ortadaki bir istek hata verirse ne olur? Sohbete yaz.